In [1]:
import pandas as pd

df = pd.read_csv("../dataset/paysim.csv")
df = df.sort_values("step", kind="stable").reset_index(drop=True)
df["orig_prev_count"] = df.groupby("nameOrig").cumcount()
df["dest_prev_count"] = df.groupby("nameDest").cumcount()

orig_prev_sum = df.groupby("nameOrig")["amount"].cumsum() - df["amount"]
df["orig_prev_avg"] = orig_prev_sum / df["orig_prev_count"].where(df["orig_prev_count"] > 0)
df["amount_vs_orig_avg"] = df["amount"] / (df["orig_prev_avg"] + 1)

df["dest_prev_received"] = df.groupby("nameDest")["amount"].cumsum() - df["amount"]

df = df[df["type"].isin(["TRANSFER", "CASH_OUT"])].copy()

# --- Features known before approval ---
df["is_transfer"] = (df["type"] == "TRANSFER").astype(int)
df["amount_to_balance"] = df["amount"] / (df["oldbalanceOrg"] + 1)
df["drains_account"] = (df["amount"] >= df["oldbalanceOrg"]).astype(int)
df["orig_zero_balance"] = (df["oldbalanceOrg"] == 0).astype(int)
df["dest_zero_balance"] = (df["oldbalanceDest"] == 0).astype(int)

FEATURES = [
    "step", "amount", "oldbalanceOrg", "oldbalanceDest",
    "is_transfer", "amount_to_balance", "drains_account",
    "orig_zero_balance", "dest_zero_balance",
    "orig_prev_count", "dest_prev_count",
    "amount_vs_orig_avg", "dest_prev_received",
]
TARGET = "isFraud"

# --- Time-based split ---
train = df[df["step"] <= 600]
test = df[df["step"] > 600]

print("train:", train.shape, "fraud:", train[TARGET].sum())
print("test: ", test.shape, "fraud:", test[TARGET].sum())

# --- Save (keep account names for the graph step) ---
keep = FEATURES + [TARGET, "nameOrig", "nameDest"]
train[keep].to_parquet("../dataset/train.parquet")
test[keep].to_parquet("../dataset/test.parquet")

train: (2726859, 21) fraud: 6613
test:  (43550, 21) fraud: 1600


In [4]:
%pip install pyarrow

   ---------------------------------------- 0.0/29.8 MB ? eta -:--:--
    --------------------------------------- 0.5/29.8 MB 3.9 MB/s eta 0:00:08
   - -------------------------------------- 1.0/29.8 MB 3.1 MB/s eta 0:00:10
   -- ------------------------------------- 1.6/29.8 MB 2.6 MB/s eta 0:00:11
   -- ------------------------------------- 1.8/29.8 MB 2.2 MB/s eta 0:00:13
   --- ------------------------------------ 2.4/29.8 MB 2.4 MB/s eta 0:00:12
   --- ------------------------------------ 2.9/29.8 MB 2.4 MB/s eta 0:00:12
   ---- ----------------------------------- 3.4/29.8 MB 2.5 MB/s eta 0:00:11
   ---- ----------------------------------- 3.7/29.8 MB 2.4 MB/s eta 0:00:11
   ----- ---------------------------------- 4.2/29.8 MB 2.3 MB/s eta 0:00:11
   ------ --------------------------------- 5.0/29.8 MB 2.4 MB/s eta 0:00:11
   ------- -------------------------------- 5.8/29.8 MB 2.6 MB/s eta 0:00:10
   -------- ------------------------------- 6.6/29.8 MB 2.6 MB/s eta 0:00:09
   ---


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pyarrow
print(pyarrow.__version__)

26.0.1


In [2]:
import os
print(os.listdir("../dataset"))

['paysim.csv', 'test.parquet', 'train.parquet']
